In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
df = pd.read_csv('salaries_dataset.csv')
df.sample(10)

,company,job,degree,salary_more_then_100k
908,google,sales executive,masters,1
214,facebook,business manager,masters,1
1400,google,computer programmer,bachelors,0
2363,facebook,computer programmer,bachelors,1
1982,facebook,computer programmer,bachelors,1
4057,google,sales executive,masters,1
3877,abc pharma,business manager,bachelors,0
1917,facebook,computer programmer,masters,0
3697,facebook,business manager,bachelors,1
2338,google,computer programmer,masters,1


In [ ]:
df.shape

(5000, 4)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 4 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   company                5000 non-null   object
 1   job                    5000 non-null   object
 2   degree                 5000 non-null   object
 3   salary_more_then_100k  5000 non-null   int64 
dtypes: int64(1), object(3)
memory usage: 156.4+ KB


In [ ]:
df.describe()

,salary_more_then_100k
count,5000.000000
mean,0.577000
std,0.494085
min,0.000000
25%,0.000000
50%,1.000000
75%,1.000000
max,1.000000


In [ ]:
df.isnull().sum()

,0
company,0
job,0
degree,0
salary_more_then_100k,0


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.feature_selection import SelectKBest,chi2
from sklearn.metrics import accuracy_score

In [ ]:
df = pd.read_csv('salaries_dataset.csv')

In [ ]:
x = df.iloc[:,0:3]
y = df.iloc[:,-1]
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42 )

In [ ]:
x_train.shape

(4000, 3)

In [ ]:
y_test.shape

(1000,)

# One Hot Encoding

In [ ]:
trf1 = ColumnTransformer([
 ('ohe_company_job',OneHotEncoder(sparse_output=False,handle_unknown='ignore'),[0,1]) ,
  ('oe_education',
     OrdinalEncoder(
         categories=[['bachelors', 'masters']]
     ),
     [2])
 ],remainder='passthrough')

In [ ]:
trf2 = ColumnTransformer([
    ('scale',MinMaxScaler(),slice(0,6))
])

In [ ]:
trf3 = SelectKBest(score_func=chi2,k=4)

In [ ]:
trf4 = DecisionTreeClassifier()

In [ ]:
pipe = Pipeline(
    [
        ('trf1',trf1),
        ('trf2',trf2),
        ('trf3',trf3),
        ('trf4',trf4)
    ]
)

In [ ]:
pipe.fit(x_train,y_train)

Pipeline(steps=[('trf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_company_job',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [0, 1]),
                                                 ('oe_education',
                                                  OrdinalEncoder(categories=[['bachelors',
                                                                              'masters']]),
                                                  [2])])),
                ('trf2',
                 ColumnTransformer(transformers=[('scale', MinMaxScaler(),
                                                  slice(0, 6, None))])),
                ('trf3',
                 SelectKBest(k=4,
                             score_func=<function chi2 at 0x7fac152ffba0>)),
                ('trf4', DecisionTreeClassifier())])

In [ ]:
y_pred = pipe.predict(x_test)

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.763

In [ ]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, y_pred)

array([[264, 143],
       [ 94, 499]])

In [ ]:
from sklearn.metrics import classification_report
classification_report(y_test, y_pred)

'              precision    recall  f1-score   support\n\n           0       0.74      0.65      0.69       407\n           1       0.78      0.84      0.81       593\n\n    accuracy                           0.76      1000\n   macro avg       0.76      0.75      0.75      1000\nweighted avg       0.76      0.76      0.76      1000\n'